New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'ai_data_scientist').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'ai_data_scientist').exists(), 'Repository root not found'
sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark' / 'projects')
from ai_data_scientist import project_manager, language_router, lifecycle
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-10-students')
lifecycle.register_run('v020-exp-10', handle.notebook_path)
lifecycle.mark_write_start('v020-exp-10')
try:
    project_manager.ensure_notebook(handle)
    project_manager.ensure_data_dir(handle)
finally:
    lifecycle.mark_write_end('v020-exp-10')
print({'root': str(ROOT), 'notebook': str(handle.notebook_path), 'language': language_router.detect_language('試験成績の差に結びつく要因を調べてください。'), 'run': lifecycle.get_run_status('v020-exp-10').state})

{'root': '/home/nahisaho/GitHub/jupytermind', 'notebook': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-10-students/notebooks/kaggle-v020-kaggle-exp-10-students.ipynb', 'language': 'ja', 'run': 'running'}


In [3]:
import nbformat, contextlib
setup_code = r'''from pathlib import Path
import os, sys, json, io, hashlib, zipfile, contextlib, warnings
from datetime import datetime, timezone
from dataclasses import asdict
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'ai_data_scientist').exists())
sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark' / 'projects')
import numpy as np
import pandas as pd
import scipy.stats as st
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from IPython.display import display
from ai_data_scientist import project_manager, language_router, lifecycle, ingestion, cleaning, eda, stats_analysis, visualization, data_definition, analysis_assumptions, data_quality, sensitivity, insight_engine, notebook_audit
RUN_ID = 'v020-exp-10'
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-10-students')
lifecycle.register_run(RUN_ID, handle.notebook_path)
DATA_DIR = project_manager.ensure_data_dir(handle)
SEED = 20261002
phase_log = []
@contextlib.contextmanager
def phase(name, write=False):
    if lifecycle.is_cancel_requested(RUN_ID):
        lifecycle.mark_failed(RUN_ID)
        raise RuntimeError('Cancellation requested')
    start = lifecycle.mark_write_start if write else lifecycle.mark_execution_start
    end = lifecycle.mark_write_end if write else lifecycle.mark_execution_end
    start(RUN_ID)
    phase_log.append({'phase': name, 'event': 'start', 'utc': datetime.now(timezone.utc).isoformat()})
    try:
        yield
    except Exception as exc:
        lifecycle.mark_failed(RUN_ID)
        phase_log.append({'phase': name, 'event': 'failed', 'exception_type': type(exc).__name__})
        raise
    finally:
        end(RUN_ID)
        phase_log.append({'phase': name, 'event': 'end', 'utc': datetime.now(timezone.utc).isoformat()})
print({'run_id': RUN_ID, 'language': language_router.detect_language('試験成績の差の要因分析'), 'seed': SEED, 'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__})
'''
acquire_code = r'''with phase('Kaggle検索・取得'):
    SLUG = 'spscientist/students-performance-in-exams'
    FILENAME = 'StudentsPerformance.csv'
    csv_path = DATA_DIR / FILENAME
    provenance_path = DATA_DIR / 'provenance.json'
    api_failure = None
    step = 'SDK import'
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi()
            step = 'authenticate'
            api.authenticate()
            step = 'dataset_list'
            matches = api.dataset_list(search='students performance in exams')
            refs = [str(d.ref) for d in matches]
            if SLUG not in refs:
                matches = api.dataset_list(search='spscientist students performance')
                refs += [str(d.ref) for d in matches]
            if SLUG not in refs:
                raise LookupError('Requested slug was not found')
            step = 'dataset_list_files'
            listing = api.dataset_list_files(SLUG)
            files = [str(f.name) for f in listing.files]
            if FILENAME not in files:
                raise FileNotFoundError('Requested CSV was not listed')
            step = 'dataset_download_file'
            cached = json.loads(provenance_path.read_text()) if provenance_path.exists() else None
            cache_ok = csv_path.exists() and cached and cached.get('source') == 'Kaggle API' and hashlib.sha256(csv_path.read_bytes()).hexdigest() == cached.get('sha256')
            if not cache_ok:
                api.dataset_download_file(SLUG, FILENAME, path=str(DATA_DIR), force=True, quiet=True)
                if not csv_path.exists():
                    archive = DATA_DIR / (FILENAME + '.zip')
                    with zipfile.ZipFile(archive) as z:
                        if FILENAME not in z.namelist():
                            raise FileNotFoundError('Expected archive member missing')
                        csv_path.write_bytes(z.read(FILENAME))
                provenance = {'source': 'Kaggle API', 'owner_dataset_slug': SLUG, 'filename': FILENAME, 'source_url': 'https://www.kaggle.com/datasets/' + SLUG, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'sha256': hashlib.sha256(csv_path.read_bytes()).hexdigest(), 'listed_files': files}
            else:
                provenance = cached
        print({'Kaggle検索結果': [r for r in refs if r == SLUG], '公開ファイル': files, '認証情報': '非表示'})
    except Exception as exc:
        # Exception messages/SDK captured output can contain credentials: never persist them.
        api_failure = {'step': step, 'exception_type': type(exc).__name__, 'http_status': getattr(exc, 'status', None), 'details': '認証情報保護のため例外本文とSDK出力は記録しない。'}
        print({'Kaggle API失敗': api_failure})
        paper = Path('/home/nahisaho/kaggle/experiments/white-paper.md').read_text(encoding='utf-8')
        FALLBACK_URL = 'https://raw.githubusercontent.com/rashida048/Datasets/master/StudentsPerformance.csv'
        assert FALLBACK_URL in paper, '旧実験のCSV URLを確認できない'
        import urllib.request
        with urllib.request.urlopen(FALLBACK_URL, timeout=60) as response:
            raw = response.read(2_000_001)
        assert len(raw) <= 2_000_000, 'Download size limit exceeded'
        csv_path.write_bytes(raw)
        provenance = {'source': '旧実験の公開CSV（フォールバック）', 'owner_dataset_slug': SLUG, 'filename': FILENAME, 'source_url': FALLBACK_URL, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'sha256': hashlib.sha256(raw).hexdigest(), 'kaggle_failure': api_failure, 'fallback_reason': 'Kaggle APIで対応する公開データを検索・取得できなかったため', 'identity_caveat': '公開CSVとKaggle掲載版の完全な同一性は保証されない'}
    with phase('来歴ファイル保存', write=True):
        provenance_path.write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding='utf-8')
    ingest_result = ingestion.ingest(ingestion.SourceSpec('csv', str(csv_path)), row_limit=10000)
    assert not ingest_result.truncated, '全件分析の行数制限を超えた'
    raw_df = ingest_result.dataframe
    display(provenance)
    display(raw_df.head())
'''
quality_code = r'''with phase('データ定義・意味検査'):
    score_cols = ['math score', 'reading score', 'writing score']
    categories = {'gender': ['female', 'male'], 'race/ethnicity': ['group A','group B','group C','group D','group E'], 'parental level of education': ["associate's degree", "bachelor's degree", 'high school', "master's degree", 'some college', 'some high school'], 'lunch': ['standard', 'free/reduced'], 'test preparation course': ['none', 'completed']}
    assert set(raw_df.columns) == set(score_cols + list(categories)), 'Unexpected or missing columns'
    schema = {c: {'min': 0, 'max': 100, 'not_null': True} for c in score_cols}
    schema.update({c: {'allowed': vals, 'not_null': True} for c, vals in categories.items()})
    semantic_findings = data_quality.detect_anomalies(raw_df, schema)
    numeric_failures = {c: int(pd.to_numeric(raw_df[c], errors='coerce').isna().sum()) for c in score_cols}
    assert not semantic_findings and not any(numeric_failures.values()), '意味的な異常を解決してから分析する必要がある'
    clean_result = cleaning.clean_dataset(raw_df, operation='drop_na')
    print('cleaning戻り値:', clean_result)
    df = raw_df.rename(columns={'gender':'gender','race/ethnicity':'ethnicity','parental level of education':'parent_education','lunch':'lunch','test preparation course':'preparation','math score':'math','reading score':'reading','writing score':'writing'}).copy()
    df['average'] = df[['math','reading','writing']].mean(axis=1)
    FV = data_definition.FieldValue
    definitions = data_definition.build_manifest(source={k: FV(v, 'verified', '取得バイト列・API応答') for k,v in provenance.items() if k in ['owner_dataset_slug','filename','retrieved_at_utc','sha256','source_url']}, dataset_scope={'row_unit': FV('生徒と推測される1行', 'inferred','列名'), 'population': FV(None,'unknown'), 'sampling_method': FV(None,'unknown'), 'country_year_school': FV(None,'unknown'), 'independent_students': FV(None,'unknown')}, variables={c:{'definition':FV('試験の'+c if c in score_cols else 'CSVのカテゴリラベル（操作的定義は未確認）','inferred','CSV列名'), 'unit':FV('0–100の数値尺度。100点満点の制度は未検証' if c in score_cols else 'カテゴリ','inferred','観測範囲'), 'measurement_protocol':FV(None,'unknown')} for c in raw_df.columns}, transformations=('欠損なしを検証、主分析では重複行を保持（生徒IDなし）','列名短縮','3科目の等重み平均（探索用の合成指標）'))
    print('意味的異常:', [asdict(x) for x in semantic_findings], '数値変換失敗:', numeric_failures)
    print('欠損セル:', int(raw_df.isna().sum().sum()), '完全一致重複行:', int(raw_df.duplicated().sum()), '0点:', {c:int((raw_df[c]==0).sum()) for c in score_cols})
    print('不明な定義:', [(p, asdict(v)) for p,v in definitions.unresolved_fields()])
    print('推測された定義:', {c:asdict(v['definition']) for c,v in definitions.variables.items()})
    display(eda.explore(raw_df))
    assumptions = analysis_assumptions.AnalysisAssumptionManifest(analysis_scope={'outcome':'3科目別および等重み平均','comparison':'Welch平均差とHC3調整回帰','target':'取得標本内の関連'}, causal_scope='associational', sampling={'n':len(df),'seed':SEED}, assumptions=(analysis_assumptions.Assumption('A1','行は独立した生徒であり、学校内クラスタ依存は無視できる','assumed',conclusion_critical=True,impact_if_false='信頼区間が狭すぎる可能性'),analysis_assumptions.Assumption('A2','等重み平均は科目横断の記述要約として使える','assumed',conclusion_critical=True,impact_if_false='科目別の差を隠す可能性。科目別感度分析を実施'),analysis_assumptions.Assumption('A3','無作為抽出と未測定交絡なしは検証されていない','rejected',conclusion_critical=True,impact_if_false='母集団への一般化と因果推論は行わない'),analysis_assumptions.Assumption('A4','欠損・範囲・カテゴリ・数値型検査を通過','verified',conclusion_critical=True)))
    assumption_findings = analysis_assumptions.check_manifest(assumptions)
    print('分析前提manifest:', asdict(assumptions))
    print('前提検査の全指摘:', [asdict(f) for f in assumption_findings])
'''
initial_code = r'''with phase('初回比較・交絡調整'):
    factors = ['gender','ethnicity','parent_education','lunch','preparation']
    rows = []
    for factor in factors:
        for label, part in df.groupby(factor, observed=True):
            rows.append({'factor':factor,'level':label,'n':len(part),'mean':part.average.mean(),'sd':part.average.std(),'se':part.average.std()/np.sqrt(len(part))})
    group_summary = pd.DataFrame(rows)
    display(group_summary.round(3))
    def welch_difference(data, factor, high, low, outcome='average'):
        a = data.loc[data[factor]==high,outcome].to_numpy()
        b = data.loc[data[factor]==low,outcome].to_numpy()
        va, vb = a.var(ddof=1)/len(a), b.var(ddof=1)/len(b)
        se = np.sqrt(va+vb)
        dof = (va+vb)**2/(va**2/(len(a)-1)+vb**2/(len(b)-1))
        diff = a.mean()-b.mean()
        ci = st.t.ppf(.975,dof)*se
        return {'factor':factor,'outcome':outcome,'contrast':high+' - '+low,'difference':diff,'ci_low':diff-ci,'ci_high':diff+ci,'p':st.ttest_ind(a,b,equal_var=False).pvalue,'standardized_difference':diff/np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))}
    crude = pd.DataFrame([welch_difference(df,'preparation','completed','none'),welch_difference(df,'lunch','standard','free/reduced'),welch_difference(df,'gender','female','male')])
    crude['p_holm'] = multipletests(crude.p, method='holm')[1]
    display(crude.round(4))
    rhs = 'C(preparation, Treatment(reference="none")) + C(lunch, Treatment(reference="free/reduced")) + C(gender, Treatment(reference="male")) + C(parent_education, Treatment(reference="high school")) + C(ethnicity, Treatment(reference="group C"))'
    model = smf.ols('average ~ '+rhs, data=df).fit(cov_type='HC3')
    adjusted = pd.DataFrame({'coefficient':model.params,'ci_low':model.conf_int()[0],'ci_high':model.conf_int()[1],'p':model.pvalues})
    adjusted['p_holm'] = np.nan
    non_intercept = adjusted.index != 'Intercept'
    adjusted.loc[non_intercept,'p_holm'] = multipletests(adjusted.loc[non_intercept,'p'],method='holm')[1]
    display(adjusted.round(4))
    print('調整モデルR2:', model.rsquared, '条件数:', model.condition_number, '最大Cook距離:', model.get_influence().cooks_distance[0].max())
    prep_term = next(t for t in model.params.index if t.startswith('C(preparation'))
    lunch_term = next(t for t in model.params.index if t.startswith('C(lunch'))
    gender_term = next(t for t in model.params.index if t.startswith('C(gender'))
    print('PRIMARY_PREP_ADJUSTED=' + format(model.params[prep_term], '.6f'))
    print('PRIMARY_LUNCH_ADJUSTED=' + format(model.params[lunch_term], '.6f'))
    print('PRIMARY_GENDER_ADJUSTED=' + format(model.params[gender_term], '.6f'))
    corr = stats_analysis.correlation(df,'reading','writing',language='ja')
    display(corr)
'''
intro = '''# Students Performance：試験成績の差に結びつく要因

run_id: `v020-exp-10`。出典：指定のKaggle公開データをAPIで検索・認証・取得し、バイト列SHA-256を保存する。認証情報は表示・保存しない。

## 分析計画（実行前）
対象は取得した全行。主指標は数学・読解・作文の等重み平均、補助指標は科目別得点。カテゴリの件数・平均・標準偏差、主要3比較（準備講座、昼食区分、gender）のWelch平均差・95%区間・標準化差を示し、3比較内のHolm補正を行う。全5要因をカテゴリとして同時投入したOLSとHC3標準誤差で観測要因を調整する。学歴を勝手に線形順序化しない。因果効果・予測ランキング・全国代表値ではない。

結果読後に科目別、相互作用・層別、モデル仕様の安定性など、結論に実質的な影響がある点を最大3回追加検討する。乱数seedを固定し、最後に新しいカーネルで全コードセルを順番に実行する。

## 限界と解釈方針
準備講座は自己選択、昼食は経済状態の不完全な代理指標になり得るが制度の詳細は未確認。事前学力、学校・教師、所得、学習時間、受講時期、欠席等がないため、交絡を取り除いたとは言えない。gender/ethnicityはデータのラベルであり生物学的・本質的説明にしない。標本設計・国・年度・生徒ID・学校クラスタ不明。HC3は異分散対応であってクラスタ依存対応ではない。信頼区間は標本が独立という作業仮定に依存する。

## 適用できない機能
独立の参照データが提供されていないため `data_quality.validate_anomalies` と `dataset_validation.compare_datasets` は適用しない。同一CSVミラーは独立検証にならない。欠損補完は欠損がある場合のみ検討する。自動機械学習・因果同定は目的外。`mcp_gateway.run_and_record` はこの対話ツールにPython MCPClientオブジェクトがないため未使用：実セルはJupyter MCPのexecute_cellで実行し、作成・追記・メタデータ書込みはenqueue_writeで直列化する。偽のexecuteアダプターで直接Python実行をMCPと称することはしない。
'''
with contextlib.ExitStack() as stack:
    lifecycle.mark_write_start('v020-exp-10')
    try:
        def initialize(nb):
            if any(c.cell_type == 'code' for c in nb.cells):
                raise RuntimeError('既存分析Notebookの上書きを拒否')
            nb.cells = [nbformat.v4.new_markdown_cell(intro), nbformat.v4.new_code_cell(setup_code), nbformat.v4.new_code_cell(acquire_code), nbformat.v4.new_code_cell(quality_code), nbformat.v4.new_code_cell(initial_code)]
            for c, role in zip(nb.cells[1:], ['setup','acquisition','quality','initial']):
                c.metadata['role'] = role
            nb.metadata['run_id'] = 'v020-exp-10'
        project_manager.enqueue_write(handle, initialize)
    finally:
        lifecycle.mark_write_end('v020-exp-10')
print('分析計画と初回実行セルを保存しました。')

分析計画と初回実行セルを保存しました。


In [8]:
iteration1_request = '3科目平均の差だけでは科目ごとの方向が隠れる可能性があります。数学・読解・作文それぞれについて準備講座、昼食区分、genderの調整差と95%信頼区間を算出し、多重比較を補正して、平均指標による結論を修正すべきか確認してください。'
iteration1_code = r'''with phase('反復1：科目別比較'):
    subject_rows = []
    for outcome in ['math','reading','writing','average']:
        fitted = smf.ols(outcome+' ~ '+rhs,data=df).fit(cov_type='HC3')
        for factor, term in [('preparation',prep_term),('lunch',lunch_term),('gender',gender_term)]:
            lo, hi = fitted.conf_int().loc[term]
            subject_rows.append({'subject':outcome,'factor':factor,'adjusted_difference':fitted.params[term],'ci_low':lo,'ci_high':hi,'p':fitted.pvalues[term]})
    subject_results = pd.DataFrame(subject_rows)
    subject_results['p_holm_12'] = multipletests(subject_results.p,method='holm')[1]
    display(subject_results)
    print('SUBJECT_GENDER_MATH=' + format(subject_results.query("subject == 'math' and factor == 'gender'").adjusted_difference.iloc[0], '.6f'))
    print('SUBJECT_GENDER_READING=' + format(subject_results.query("subject == 'reading' and factor == 'gender'").adjusted_difference.iloc[0], '.6f'))
    print('SUBJECT_GENDER_WRITING=' + format(subject_results.query("subject == 'writing' and factor == 'gender'").adjusted_difference.iloc[0], '.6f'))
    print('反復1判断：科目別の方向を読み、平均のみのgender結論は採用しない。多重補正は12係数内の探索的補正で、研究全体の選択過程を保証しない。')
'''
with phase('反復1セル書込み', write=True):
    def add_iteration1(nb):
        for c in nb.cells:
            if c.metadata.get('role') == 'quality':
                c.source = c.source.replace("print('cleaning戻り値:', clean_result)", "print('欠損除去影響:', {'rows_before': len(raw_df), 'rows_after': len(clean_result.dataframe), 'columns_before': len(raw_df.columns), 'columns_after': len(clean_result.dataframe.columns)})")
            if c.metadata.get('role') == 'setup':
                c.source = c.source.replace("phase_log = []", "phase_log = []\npd.set_option('display.max_colwidth', 120)\npd.set_option('display.max_columns', 12)\npd.set_option('display.float_format', lambda v: f'{v:.6g}')")
        nb.cells.append(nbformat.v4.new_markdown_cell('# 反復依頼履歴：第1回\n\n原文：\n\n'+iteration1_request+'\n\n選定理由：初回のgender平均差は科目の構成による可能性があり、結論の方向に直結する。証拠は直後の実行セル。残る限界：科目間の測定尺度と観測されない事前学力は不明。'))
        c = nbformat.v4.new_code_cell(iteration1_code)
        c.metadata['role']='iteration1'
        nb.cells.append(c)
    project_manager.enqueue_write(handle, add_iteration1)
print('反復1を追加しました。')

反復1を追加しました。


In [10]:
iteration2_request = '準備講座の完了者は自己選択された集団です。受講割合を観測属性別に比較し、昼食区分×genderの4層で調整済み成績差と人数を確認してください。準備講座との交互作用も検定し、全員に同じ差があるという結論が妥当か評価してください。'
iteration2_code = r'''with phase('反復2：自己選択・層別'):
    selection_rows = []
    for factor in ['lunch','gender','parent_education','ethnicity']:
        for label, part in df.groupby(factor,observed=True):
            selection_rows.append({'factor':factor,'level':label,'n':len(part),'completed_fraction':part.preparation.eq('completed').mean()})
    selection = pd.DataFrame(selection_rows)
    display(selection)
    strata_rows = []
    for (lunch, gender), part in df.groupby(['lunch','gender'],observed=True):
        fitted = smf.ols('average ~ C(preparation, Treatment(reference="none")) + C(parent_education, Treatment(reference="high school")) + C(ethnicity, Treatment(reference="group C"))',data=part).fit(cov_type='HC3')
        lo,hi = fitted.conf_int().loc[prep_term]
        strata_rows.append({'lunch':lunch,'gender':gender,'n_completed':int(part.preparation.eq('completed').sum()),'n_none':int(part.preparation.eq('none').sum()),'difference':fitted.params[prep_term],'ci_low':lo,'ci_high':hi,'p':fitted.pvalues[prep_term]})
    strata = pd.DataFrame(strata_rows)
    strata['p_holm_4'] = multipletests(strata.p,method='holm')[1]
    display(strata)
    interaction_model = smf.ols('average ~ '+rhs+' + C(preparation, Treatment(reference="none")):C(lunch, Treatment(reference="free/reduced")) + C(preparation, Treatment(reference="none")):C(gender, Treatment(reference="male"))',data=df).fit(cov_type='HC3')
    interaction_terms = [t for t in interaction_model.params.index if ':' in t]
    interactions = pd.DataFrame({'difference_in_differences':interaction_model.params[interaction_terms],'ci_low':interaction_model.conf_int().loc[interaction_terms,0],'ci_high':interaction_model.conf_int().loc[interaction_terms,1],'p':interaction_model.pvalues[interaction_terms]})
    interactions['p_holm_2'] = multipletests(interactions.p,method='holm')[1]
    display(interactions)
    print('STRATA_PREP_MIN=' + format(strata.difference.min(), '.6f'))
    print('STRATA_PREP_MAX=' + format(strata.difference.max(), '.6f'))
    print('INTERACTION_MIN_ADJUSTED_P=' + format(interactions.p_holm_2.min(), '.6f'))
    print('反復2の限界：層別は観測した2属性のみ。非有意の交互作用は差が同一という証明ではない。事前学力・意欲による自己選択は検証不能。')
'''
with phase('反復2書込み',write=True):
    def append2(nb):
        nb.cells.append(nbformat.v4.new_markdown_cell('# 反復依頼履歴：第2回\n\n原文：\n\n'+iteration2_request+'\n\n選定理由：科目別でも準備講座の差は正だったが、自己選択や属性による差の異質性が未解決。証拠は受講割合・層別表・交互作用係数。残る限界：未測定交絡、層別の小標本、探索的検定。'))
        c=nbformat.v4.new_code_cell(iteration2_code); c.metadata['role']='iteration2'; nb.cells.append(c)
    project_manager.enqueue_write(handle,append2)
print('反復2を追加しました。')

反復2を追加しました。


In [12]:
iteration3_request = '層別でも準備講座の正の関連が残りましたが、モデル仕様や影響の強い行に依存していないか、OLSと中央値回帰、全行とCook距離による感度用除外、ethnicity調整の有無を比較してください。さらに未測定交絡が主な差を打ち消す強さを線形回帰の部分R2シナリオで示し、頑健性を因果効果の証明と誤解しないよう結論を整理してください。'
iteration3_code = r'''with phase('反復3：仕様と未測定交絡への感度'):
    cook = model.get_influence().cooks_distance[0]
    influence_keep = cook <= 4 / len(df)
    print('Cook感度用除外数:', int((~influence_keep).sum()), '重複を除いた行数:', len(raw_df.drop_duplicates()))
    rhs_without_ethnicity = rhs.rsplit(' + C(ethnicity',1)[0]
    sensitivity_plan = sensitivity.SensitivityPlan(parameter_grid={'model_kind':['ols','median'],'subset':['all','low_influence'],'adjustment':['full','no_ethnicity']},max_runs=8)
    fitted_specs = {}
    def specification_fit(model_kind, subset, adjustment):
        key=(model_kind,subset,adjustment)
        if key not in fitted_specs:
            part=df if subset=='all' else df.loc[influence_keep]
            formula='average ~ '+(rhs if adjustment=='full' else rhs_without_ethnicity)
            if model_kind=='ols':
                fitted_specs[key]=smf.ols(formula,data=part).fit(cov_type='HC3')
            else:
                fitted_specs[key]=smf.quantreg(formula,data=part).fit(q=.5,max_iter=5000,p_tol=1e-7)
                assert fitted_specs[key].iterations < 5000, '中央値回帰が未収束'
        return fitted_specs[key]
    sensitivity_reports = {}
    for factor,term in [('preparation',prep_term),('lunch',lunch_term)]:
        def analyze_spec(model_kind,subset,adjustment,term=term):
            return specification_fit(model_kind,subset,adjustment).params[term]
        report=sensitivity.run_sensitivity(sensitivity_plan,analyze_spec,stability_tolerance=.20)
        assert all(np.isfinite(r.value) for r in report.results), 'Nonfinite sensitivity result'
        sensitivity_reports[factor]=report
        display(pd.DataFrame([{**r.specification,'factor':factor,'coefficient':r.value} for r in report.results]))
        print(f'SENSITIVITY_{factor.upper()}=' + json.dumps({'stable':report.stable,'max_relative_deviation':report.max_relative_deviation,'positive_in_all_specs':all(r.value>0 for r in report.results)},sort_keys=True))
    # Point-estimate omitted-variable bias scenario for a linear model, not causal identification.
    classical = smf.ols('average ~ '+rhs,data=df).fit()
    ovb_rows = []
    robustness_rows=[]
    for factor,term in [('preparation',prep_term),('lunch',lunch_term)]:
        f2 = classical.tvalues[term]**2/classical.df_resid
        rv = (np.sqrt(f2**2+4*f2)-f2)/2
        robustness_rows.append({'factor':factor,'equal_partial_R2_to_zero_point_estimate':rv,'classical_residual_df':classical.df_resid})
        for r in [.05,.10,.20,.30]:
            bias = classical.bse[term]*np.sqrt(classical.df_resid*r*r/(1-r))
            ovb_rows.append({'factor':factor,'partial_R2_D_U_given_X':r,'partial_R2_Y_U_given_D_X':r,'worst_direction_bias':bias,'adjusted_point_estimate':classical.params[term]-bias})
    robustness=pd.DataFrame(robustness_rows)
    ovb_scenarios=pd.DataFrame(ovb_rows)
    display(robustness)
    display(ovb_scenarios)
    print('OVB_PREP_TIPPING_R2=' + format(robustness.query("factor == 'preparation'").equal_partial_R2_to_zero_point_estimate.iloc[0], '.6f'))
    print('OVB_LUNCH_TIPPING_R2=' + format(robustness.query("factor == 'lunch'").equal_partial_R2_to_zero_point_estimate.iloc[0], '.6f'))
    print('反復3の限界：20%の許容差は事前の科学的閾値ではなく、仕様比較の明示的な実務上の目安。中央値回帰は平均差と異なる推定対象。Cook除外は感度用で主分析は全行を保持。')
    print('未測定交絡シナリオ：bias = SE_classical * sqrt(df_resid * R2_YU|DX * R2_DU|X / (1-R2_DU|X))。等しいR2の点推定ゼロ化閾値RV=(sqrt(f2^2+4*f2)-f2)/2、f2=t_classical^2/df_resid。参考：Cinelli & Hazlett (2020), JRSS B, doi:10.1111/rssb.12348。HC3の有意性を消す閾値ではなく、交絡の実際の強さも不明。')
'''
with phase('反復3書込み',write=True):
    def append3(nb):
        nb.cells.append(nbformat.v4.new_markdown_cell('# 反復依頼履歴：第3回\n\n原文：\n\n'+iteration3_request+'\n\n選定理由：観測層別だけでは残る自己選択を評価できず、仕様依存と未測定交絡への感度は結論の強さを左右する。証拠は8仕様比較と部分R2シナリオ。残る限界：学校内依存・抽出機構・事前学力はデータから追加検証不能。最大3回で停止し、未知の因果構造を追加計算だけで解消したとは主張しない。'))
        c=nbformat.v4.new_code_cell(iteration3_code); c.metadata['role']='iteration3'; nb.cells.append(c)
    project_manager.enqueue_write(handle,append3)
print('反復3を追加しました。')

反復3を追加しました。


In [14]:
chart_code = r'''with phase('図表作成'):
    chart_checks=[]
    def show_png(png, name, caught):
        missing=[str(w.message) for w in caught if 'Glyph' in str(w.message) and 'missing' in str(w.message)]
        assert not missing, '欠落した文字グリフがある'
        chart_checks.append({'chart':name,'missing_glyphs':missing,'png_bytes':len(png)})
        display(visualization.build_image_output(png)['data'],raw=True)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        bar_data=df.groupby('preparation',observed=True).average.mean().rename('3科目平均').reset_index()
        png=visualization.render_chart(bar_data,kind='bar',x='preparation',y='3科目平均',title='準備講座区分別の3科目平均（未調整）',xlabel='準備講座区分',ylabel='平均得点（観測尺度）')
    show_png(png,'未調整平均',caught)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        fig,axes=plt.subplots(1,3,figsize=(13,4),sharex=True)
        label_map={'preparation':'completed − none','lunch':'standard − free/reduced','gender':'female − male'}
        colors=['#0072B2','#D55E00','#009E73']
        for ax,(factor,color) in zip(axes,zip(['preparation','lunch','gender'],colors)):
            part=subject_results.query('factor == @factor').set_index('subject').loc[['math','reading','writing','average']]
            y=np.arange(len(part))
            ax.errorbar(part.adjusted_difference,y,xerr=np.vstack([part.adjusted_difference-part.ci_low,part.ci_high-part.adjusted_difference]),fmt='o',color=color,capsize=4,label='調整差と95% HC3区間')
            ax.axvline(0,color='gray',ls='--')
            ax.set_yticks(y,['数学','読解','作文','3科目平均']); ax.invert_yaxis()
            ax.set_title(factor+'\n'+label_map[factor]); ax.set_xlabel('観測尺度上の調整差'); ax.set_ylabel('科目')
            ax.legend(fontsize=8,loc='lower right'); ax.grid(axis='x',alpha=.2)
        fig.suptitle('科目別の関連：同じ5要因を調整（因果効果ではない）')
        fig.tight_layout(); buf=io.BytesIO(); fig.savefig(buf,format='png',dpi=140); plt.close(fig)
    show_png(buf.getvalue(),'科目別調整差',caught)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        fig,ax=plt.subplots(figsize=(9,4))
        names=[f'{r.lunch} / {r.gender} (n={r.n_completed}+{r.n_none})' for r in strata.itertuples()]
        ax.errorbar(strata.difference,np.arange(len(strata)),xerr=np.vstack([strata.difference-strata.ci_low,strata.ci_high-strata.difference]),fmt='o',capsize=4,label='準備講座の調整差と95% HC3区間')
        ax.set_yticks(np.arange(len(strata)),names); ax.invert_yaxis(); ax.axvline(0,color='gray',ls='--')
        ax.set_title('昼食区分×gender別の準備講座の関連'); ax.set_xlabel('completed − none：3科目平均の調整差'); ax.set_ylabel('層（完了者数＋未完了者数）'); ax.legend(loc='lower right'); ax.grid(axis='x',alpha=.2)
        fig.tight_layout(); buf=io.BytesIO(); fig.savefig(buf,format='png',dpi=140); plt.close(fig)
    show_png(buf.getvalue(),'層別調整差',caught)
    print('図表生成検査:',chart_checks)
'''
ledger_code = r'''with phase('証拠台帳・再現性基準'):
    evidence_values = {
        'n':str(len(df)), 'prep_adjusted':format(model.params[prep_term],'.6f'), 'lunch_adjusted':format(model.params[lunch_term],'.6f'),
        'prep_ci':f'{model.conf_int().loc[prep_term,0]:.6f}, {model.conf_int().loc[prep_term,1]:.6f}',
        'lunch_ci':f'{model.conf_int().loc[lunch_term,0]:.6f}, {model.conf_int().loc[lunch_term,1]:.6f}',
        'gender_math':format(subject_results.query("subject == 'math' and factor == 'gender'").adjusted_difference.iloc[0],'.6f'),
        'gender_reading':format(subject_results.query("subject == 'reading' and factor == 'gender'").adjusted_difference.iloc[0],'.6f'),
        'gender_writing':format(subject_results.query("subject == 'writing' and factor == 'gender'").adjusted_difference.iloc[0],'.6f'),
        'strata_min':format(strata.difference.min(),'.6f'), 'strata_max':format(strata.difference.max(),'.6f'),
        'interaction_p':format(interactions.p_holm_2.min(),'.6f'),
        'sensitivity_prep':format(sensitivity_reports['preparation'].max_relative_deviation,'.6f'),
        'sensitivity_lunch':format(sensitivity_reports['lunch'].max_relative_deviation,'.6f'),
        'rv_prep':format(robustness.iloc[0].equal_partial_R2_to_zero_point_estimate,'.6f'),
        'rv_lunch':format(robustness.iloc[1].equal_partial_R2_to_zero_point_estimate,'.6f'),
        'reading_writing_corr':format(corr.statistic,'.6f'), 'R2':format(model.rsquared,'.6f'),
    }
    edu_terms=[t for t in model.params.index if t.startswith('C(parent_education')]
    education_results=adjusted.loc[edu_terms].copy()
    display(education_results)
    master_term=next(t for t in edu_terms if "master's degree" in t)
    evidence_values['master_vs_high_school']=format(model.params[master_term],'.6f')
    # Rich text output is directly supported by the evidence engine; no fabricated execution output.
    display({'text/plain':json.dumps(evidence_values,ensure_ascii=False,indent=2)},raw=True)
    metrics=np.concatenate([model.params.to_numpy(),subject_results.adjusted_difference.to_numpy(),strata.difference.to_numpy(),*[np.array([r.value for r in report.results]) for report in sensitivity_reports.values()]])
    baseline_path=DATA_DIR/'reproducibility-baseline.json'
    if baseline_path.exists():
        baseline=json.loads(baseline_path.read_text())
        assert baseline['sha256']==provenance['sha256'], 'データのバイト列が初回と異なる'
        assert np.allclose(metrics,baseline['metrics'],rtol=1e-8,atol=1e-8), '初回と再実行の主な推定値が一致しない'
        reproduction={'baseline_comparison':'passed','metric_count':len(metrics),'max_absolute_difference':float(np.max(np.abs(metrics-np.asarray(baseline['metrics'])))),'rtol':1e-8,'atol':1e-8,'sha256_match':True}
    else:
        with phase('再現性基準保存',write=True):
            baseline_path.write_text(json.dumps({'sha256':provenance['sha256'],'metrics':metrics.tolist()}),encoding='utf-8')
        reproduction={'baseline_comparison':'baseline_created','metric_count':len(metrics),'clean_replay':'pending'}
    print('再現性比較:',reproduction)
'''
modules_md = '''# 使用したJupytermindモジュール

直接importした `ai_data_scientist` の以下のサブモジュールのみを記載する。間接利用・未実行機能は含めない。

| モジュール | 直接呼び出す関数・クラス | 使用目的 |
|---|---|---|
| project_manager | resolve_project, ensure_notebook（作成用Notebook）, ensure_data_dir, enqueue_write | slug検証、安定した保存先、直列書込み |
| language_router | detect_language | 日本語判定 |
| lifecycle | register_run, mark_execution_start/end, mark_write_start/end, is_cancel_requested, mark_completed, mark_failed（失敗時）, get_run_status, wait_for_quiescence | 実行・書込み・終了状態 |
| ingestion | SourceSpec, ingest | CSV読込みと行数制限 |
| cleaning | clean_dataset | 欠損除去の影響確認（欠損0なので変更0） |
| eda | explore | 型・欠損・カテゴリ・数値要約 |
| stats_analysis | correlation | 読解と作文の相関 |
| visualization | render_chart, build_image_output | 日本語対応の描画と画像MIME |
| data_definition | FieldValue, build_manifest, DataDefinitionManifest.unresolved_fields | 出典・定義と不確実性 |
| analysis_assumptions | Assumption, AnalysisAssumptionManifest, check_manifest | 仮定・関連に限定した解釈 |
| data_quality | detect_anomalies | 範囲・カテゴリ・欠損の意味検査 |
| sensitivity | SensitivityPlan, run_sensitivity | 各要因8仕様の比較 |
| insight_engine | extract_cited_value, record_insight | 実出力からの根拠抽出とInsight登録 |
| notebook_audit | audit_notebook | 実行・証拠・visual_audit=True監査 |

`mcp_gateway`、`dataset_validation`、`anomaly_detection`などは直接利用していない。実際の分析・APIコードはJupyter MCPのexecute_cellのみで実行した。statsmodels等はJupytermindではなく外部Pythonライブラリである。
'''
with phase('図表・証拠台帳書込み',write=True):
    def append_outputs(nb):
        nb.cells.append(nbformat.v4.new_markdown_cell('# 図表\n\n調整差の区間は各係数の95% HC3区間であり同時信頼区間ではない。多重比較は表のHolm補正p値で別途評価する。PNGは埋込み保存し、実グリフ警告・タイトル・軸・凡例・画像密度を監査する。'))
        c=nbformat.v4.new_code_cell(chart_code); c.metadata.update({'role':'charts','chart':{'title':'未調整平均、科目別調整差、層別調整差','xlabel':'準備講座区分または観測尺度上の調整差','ylabel':'平均得点、科目、層','legend':'平均・調整差と95% HC3区間','missing_glyphs':[]}}); nb.cells.append(c)
        c=nbformat.v4.new_code_cell(ledger_code); c.metadata['role']='ledger'; nb.cells.append(c)
        nb.cells.append(nbformat.v4.new_markdown_cell(modules_md))
        # Tables must not hide reference-level labels or make small p-values look exactly zero.
        for c in nb.cells:
            if c.metadata.get('role')=='initial':
                c.source=c.source.replace('display(crude.round(4))','display(crude)').replace('display(adjusted.round(4))','display(adjusted)')
    project_manager.enqueue_write(handle,append_outputs)
print('図表・証拠台帳セルを保存しました。')

図表・証拠台帳セルを保存しました。


In [17]:
import base64
snapshot=nbformat.read(handle.notebook_path,as_version=4)
chart_cell=next(c for c in snapshot.cells if c.metadata.get('role')=='charts')
original_png=next(o['data']['image/png'] for o in chart_cell.outputs if 'image/png' in o.get('data',{}))
(DATA_DIR/'visual-review-original-bar.png').write_bytes(base64.b64decode(original_png))
final_code = r'''import nbformat
with phase('根拠付きInsight・manifest・改善候補の保存',write=True):
    snapshot=nbformat.read(handle.notebook_path,as_version=4)
    initial_cell=next(c for c in snapshot.cells if c.metadata.get('role')=='initial')
    ledger_cell=next(c for c in snapshot.cells if c.metadata.get('role')=='ledger')
    ledger_output='\n'.join(str(o.get('data',{}).get('text/plain','')) for o in ledger_cell.outputs)
    def cited(key):
        return insight_engine.extract_cited_value({'output':ledger_output},r'"'+key+r'": "([^"]+)"')
    stream_text=''.join(o.get('text','') for o in initial_cell.outputs if o.output_type=='stream')
    stream_value=insight_engine.extract_cited_value({'output':stream_text},r'PRIMARY_PREP_ADJUSTED=([-0-9.]+)')
    try:
        insight_engine.record_insight(handle,'ストリーム出力検証用（成功時のみ記録）',initial_cell.execution_count,stream_value,'associational',language='ja')
    except insight_engine.EvidenceMissingError:
        stream_issue={'classification':'機能不足・証拠出力形式','reproduction':'初回分析セルのprint(PRIMARY_PREP_ADJUSTED=...)の値を実execution_countでrecord_insightに渡す','expected':'実行済みstream.text内の値も根拠として認識する','actual':'EvidenceMissingError。data MIMEだけが探索されstream.textは認識されない','impact':'正しく実行されたprint出力からInsightを記録できない','workaround':'証拠台帳セルで同じ計算値を実際のdisplay text/plainとして出力し、この出力だけを引用','related_cells':['initial','ledger','final'],'log':'この最終セルのSTREAM_EVIDENCE_REPROとdata/jupytermind-improvement-candidates.json'}
        print('STREAM_EVIDENCE_REPRO=EvidenceMissingError (expected reproduction; rich output workaround used)')
    else:
        stream_issue={'classification':'なし','actual':'stream形式はこの環境で認識された'}
    improvements=[stream_issue,{'classification':'可視化・ラベル切れの疑い','reproduction':"visualization.render_chart(kind='bar', x='preparation')に長いカテゴリcompleted/noneを渡す（元画像保存済み）",'expected':'PNGの全ラベルがキャンバス内に収まる','actual':'元のPNGで回転したcompletedの下部と横軸ラベルが切れた。欠落グリフ警告や画像密度監査では検出されない','impact':'図表の読解性低下、visual_audit=Trueだけでは見逃す','workaround':'ライブラリ図のカテゴリ表示を短い日本語に変更。カスタム図にはbbox_inches=tight、凡例を外側に配置し再描画','related_cells':['charts','final'],'log':'data/visual-review-original-bar.png と修正後のcharts出力'}]
    def remove_old(nb):
        nb.cells=[c for c in nb.cells if not c.metadata.get('generated_insight') and not c.source.startswith('ストリーム出力検証用')]
    project_manager.enqueue_write(handle,remove_old)
    claims=[
        ('prep_adjusted',f'準備講座の完了者は未完了者より3科目平均が調整後 {cited("prep_adjusted")} 高い（95% HC3区間 {cited("prep_ci")}）。5要因を同時に調整しても関連が残るが、事前学力・意欲・受講時期の自己選択は未測定であり講座の効果ではない。'),
        ('lunch_adjusted',f'lunch=standardはfree/reducedより調整後 {cited("lunch_adjusted")} 高い（95% HC3区間 {cited("lunch_ci")}）。経済環境との関連を疑う手がかりだが、昼食制度・所得・学校環境が不明で給食変更による効果とは言えない。'),
        ('gender_math',f'第1回反復で平均のみのgender解釈を修正した。female−maleの調整差は数学 {cited("gender_math")}、読解 {cited("gender_reading")}、作文 {cited("gender_writing")} と方向が異なる。これは標本内の科目別関連であり、本質的能力差を示さない。'),
        ('master_vs_high_school',f'保護者学歴のmaster\'s degreeとhigh schoolの調整差は {cited("master_vs_high_school")}。カテゴリを順序尺度にせず推定した関連で、学歴の介入効果ではない。学歴・ethnicityの各比較は初回表のHolm補正で評価し、少数カテゴリや基準カテゴリ選択にも注意する。'),
        ('strata_min',f'第2回反復：準備講座の4層別調整差は {cited("strata_min")}〜{cited("strata_max")} で全層正だった。2交互作用の最小Holm補正p値は {cited("interaction_p")}。異質性の証拠は弱いが、差が同じと証明したわけではなく未測定の自己選択も残る。'),
        ('sensitivity_prep',f'第3回反復：各8仕様で準備講座・昼食の正の関連は維持された。最大相対偏差はそれぞれ {cited("sensitivity_prep")}、{cited("sensitivity_lunch")}、両report.stable=True（明示した20%目安内）。点推定をゼロにする等強度の部分R2シナリオは準備講座 {cited("rv_prep")}、昼食 {cited("rv_lunch")}。仕様安定性は未測定交絡なしを保証せず、有意性を消す閾値でもない。'),
        ('reading_writing_corr',f'読解と作文の相関は {cited("reading_writing_corr")} と強い正の関連。3科目平均は独立な3測定を平均しているのではなく、言語科目を二重に重み付けする面がある。相関のp=0表示は浮動小数点での極小p値で、真の確率ゼロとは解釈しない。'),
        ('R2',f'全5要因モデルのR2は {cited("R2")}。取得標本 {cited("n")} 行内での説明的要約に留まる。標本抽出・学校クラスタ・事前学力・試験定義が不明なため全国への一般化や因果同定は行わない。第3回で停止：残る問題は追加の同じCSV計算では解消できず、独立データと事前学力・学校情報が必要。'),
    ]
    for key,text in claims:
        insight_engine.record_insight(handle,text,ledger_cell.execution_count,cited(key),'associational',language='ja')
        def tag_latest(nb):
            nb.cells[-1].metadata['generated_insight']=True
        project_manager.enqueue_write(handle,tag_latest)
    previous_counts=[c.execution_count for c in snapshot.cells if c.cell_type=='code' and c.metadata.get('role')!='final']
    fresh_sequence=previous_counts==list(range(1,len(previous_counts)+1)) and get_ipython().execution_count==len(previous_counts)+1
    reproduction['clean_kernel_top_to_bottom']=fresh_sequence
    reproduction['previous_code_execution_counts']=previous_counts
    reproduction['current_execution_count']=get_ipython().execution_count
    if fresh_sequence:
        assert reproduction['baseline_comparison']=='passed'
    with phase('manifestとログの保存',write=True):
        for filename,payload in [('data-definition-manifest.json',asdict(definitions)),('analysis-assumptions-manifest.json',asdict(assumptions)),('jupytermind-improvement-candidates.json',improvements),('reproducibility-verification.json',reproduction)]:
            (DATA_DIR/filename).write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
    def finish_layout(nb):
        final=next(c for c in nb.cells if c.metadata.get('role')=='final')
        nb.cells=[c for c in nb.cells if c is not final]+[final]
        nb.metadata['reproducibility']=reproduction
        nb.metadata['jupytermind_improvement_candidates']=improvements
        nb.metadata['chart_manual_review']='元のbarの軸ラベル切れとカスタム図の凡例重なりを目視確認。短いカテゴリ・凡例外側配置で修正し再描画。'
    project_manager.enqueue_write(handle,finish_layout)
    print('Jupytermind改善候補:',json.dumps(improvements,ensure_ascii=False,indent=2))
    print('他層の問題との切り分け：最初の保存先ディレクトリ未作成はJupyter MCP利用手順、filesystem MCPの許可ルート不一致はMCP設定、作成用セルのcontextlib import漏れは分析者コード。Kaggle API失敗なし、ベンチマークランナー未使用。これらはJupytermind不具合に含めない。')
with phase('notebook_audit（visual_audit=True）'):
    audit=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    print('notebook_audit:',json.dumps({**asdict(audit),'ok':audit.ok},ensure_ascii=False,indent=2))
    assert audit.ok, 'Notebook audit failed'
with phase('監査報告の書込み',write=True):
    def store_audit(nb):
        nb.metadata['audit']={**asdict(audit),'ok':audit.ok}
    project_manager.enqueue_write(handle,store_audit)
lifecycle.mark_completed(RUN_ID)
status=lifecycle.wait_for_quiescence(RUN_ID)
assert status.state=='completed' and status.active_cell_executions==status.pending_notebook_writes==status.locks_held==0
(DATA_DIR/'lifecycle-log.json').write_text(json.dumps({'events':phase_log,'final_status':asdict(status)},ensure_ascii=False,indent=2),encoding='utf-8')
print('再実行検証:',json.dumps(reproduction,ensure_ascii=False,indent=2))
print('lifecycle最終状態:',asdict(lifecycle.get_run_status(RUN_ID)))
'''
with phase('最終セル・図表修正の書込み',write=True):
    def add_final(nb):
        for c in nb.cells:
            role=c.metadata.get('role')
            if role=='charts':
                c.source=c.source.replace("png=visualization.render_chart(bar_data", "bar_data['preparation']=bar_data.preparation.map({'completed':'完了','none':'未完了'})\n        png=visualization.render_chart(bar_data")
                c.source=c.source.replace("ax.legend(fontsize=8,loc='lower right')", "ax.legend(fontsize=8,loc='upper center',bbox_to_anchor=(.5,-.25))")
                c.source=c.source.replace("ax.legend(loc='lower right')", "ax.legend(loc='upper center',bbox_to_anchor=(.5,-.25))")
                c.source=c.source.replace("fig.savefig(buf,format='png',dpi=140)", "fig.savefig(buf,format='png',dpi=140,bbox_inches='tight')")
            if role=='initial':
                c.source=c.source.replace('display(adjusted)', "display(adjusted.rename(index=lambda s: s if s=='Intercept' else s.split('(')[0]+': '+s.split('[T.')[-1].removesuffix(']')))")
            if role=='iteration2':
                c.source=c.source.replace('display(interactions)', "display(interactions.rename(index=lambda s: 'prep × '+('lunch' if ':C(lunch' in s else 'gender'))))")
            if role=='ledger':
                c.source=c.source.replace('display(education_results)', "display(education_results.rename(index=lambda s: s.split('[T.')[-1].removesuffix(']')+' − high school'))")
        nb.cells.append(nbformat.v4.new_markdown_cell('# Jupytermind改善候補\n\n分類・再現条件・期待と実際の挙動・影響・回避策・関係セルとログを最終セルの出力、および `data/jupytermind-improvement-candidates.json` に記録する。ストリーム根拠の機能不足は実セルで再現し、可視化ラベル切れは元PNGを保存した。コード・設定・Kaggle・ベンチマーク由来の問題は分離する。\n\n## 結果に対する図表の目視監査\n元のbarでは長い回転カテゴリと軸ラベルが切れたため短い日本語表記へ変更。科目別・層別の区間と凡例が重なったため凡例を外側へ移し、tight bounding boxで保存する。visual_auditの画像密度は画素内容の厳密監査ではないので、生成後の実画像も確認する。'))
        nb.cells.append(nbformat.v4.new_markdown_cell('# 根拠付きInsightと再実行・監査\n\n最終セルは実行済み証拠台帳から引用値を抽出し、record_insightで登録する。再実行時は生成Insightのみ置換して重複と古いexecution_countを防ぐ。最後の監査セルの実行中に限る監査上の例外があるため、完了後にも作成用NotebookのJupyter MCPセルから読取り専用監査を実施する。再現にはPython依存環境、Kaggle認証、ネットワークが必要。再取得を省く際は初回CSVのSHA-256を検証したうえで、API認証・検索・ファイル一覧は再実行する。'))
        c=nbformat.v4.new_code_cell(final_code);c.metadata['role']='final';nb.cells.append(c)
    project_manager.enqueue_write(handle,add_final)
print('最終監査セルと図表修正を保存しました。')

最終監査セルと図表修正を保存しました。


In [19]:
with phase('軸ラベルの余白修正',write=True):
    def chart_layout(nb):
        for c in nb.cells:
            if c.metadata.get('role')=='charts':
                c.source=c.source.replace('with warnings.catch_warnings(record=True) as caught:', "with warnings.catch_warnings(record=True) as caught, plt.rc_context({'figure.autolayout': True}):")
            if c.metadata.get('role')=='final':
                c.source=c.source.replace('ライブラリ図のカテゴリ表示を短い日本語に変更。', 'ライブラリ図はfigure.autolayout=Trueのrc_contextで余白調整し、カテゴリ表示も短い日本語に変更。')
            if c.cell_type=='markdown' and c.source.startswith('# Jupytermind改善候補'):
                c.source=c.source.replace('短い日本語表記へ変更。', '短い日本語表記へ変更し、figure.autolayout=Trueで余白を調整。')
    project_manager.enqueue_write(handle,chart_layout)
print('画像の実表示を読み直し、barの軸ラベルの余白も修正しました。')

画像の実表示を読み直し、barの軸ラベルの余白も修正しました。


In [22]:
with phase('再実行用Insight位置固定',write=True):
    def stabilize_insights(nb):
        generated=[c for c in nb.cells if c.metadata.get('generated_insight')]
        assert len(generated)==8
        for i,c in enumerate(generated):
            c.metadata['insight_slot']=i
        final=next(c for c in nb.cells if c.metadata.get('role')=='final')
        final.source=final.source.replace("    def remove_old(nb):\n        nb.cells=[c for c in nb.cells if not c.metadata.get('generated_insight') and not c.source.startswith('ストリーム出力検証用')]\n    project_manager.enqueue_write(handle,remove_old)", "    assert sum(bool(c.metadata.get('generated_insight')) for c in snapshot.cells)==8, 'Insightスロットが不足'")
        final.source=final.source.replace('    for key,text in claims:', '    for slot,(key,text) in enumerate(claims):')
        final.source=final.source.replace("        def tag_latest(nb):\n            nb.cells[-1].metadata['generated_insight']=True", "        def tag_latest(nb):\n            generated=nb.cells.pop()\n            generated.metadata.update({'generated_insight':True,'insight_slot':slot})\n            index=next(i for i,c in enumerate(nb.cells) if c.metadata.get('insight_slot')==slot)\n            nb.cells[index]=generated")
        final.source=final.source.replace("        final=next(c for c in nb.cells if c.metadata.get('role')=='final')\n        nb.cells=[c for c in nb.cells if c is not final]+[final]", "        assert nb.cells[-1].metadata.get('role')=='final', '最終セルの位置が変わった'")
        final.source=final.source.replace('これらはJupytermind不具合に含めない。', '実行中セルを移動するとMCP出力が位置追従せず最終セルが未実行表示になった事象も、MCPと書込み手順の相互作用として分離。Insightスロットの位置を固定して回避。これらはJupytermind不具合に含めない。')
    project_manager.enqueue_write(handle,stabilize_insights)
print('実行中のセル位置を変えずにInsightを更新するよう固定しました。', lifecycle.get_run_status(RUN_ID).state)

実行中のセル位置を変えずにInsightを更新するよう固定しました。 completed


In [24]:
lifecycle.register_run(RUN_ID,handle.notebook_path)
with phase('クリーン再実行の準備',write=True):
    def prepare_clean_replay(nb):
        for c in nb.cells:
            if c.cell_type=='code':
                c.execution_count=None
                c.outputs=[]
            if c.metadata.get('role')=='setup':
                c.source=c.source.replace('from dataclasses import asdict', 'from dataclasses import asdict\nfrom importlib.metadata import version')
                c.source += "\nprint('実行依存環境:', {p:version(p) for p in ['scipy','statsmodels','matplotlib','kaggle','nbformat','japanize-matplotlib']})\n"
            if c.cell_type=='markdown' and c.source.startswith('# 使用したJupytermindモジュール'):
                c.source=c.source.replace(', mark_failed（失敗時）','')
                c.source += '\n\n`lifecycle.mark_failed` は失敗分岐に記述したが成功した分析・再実行では呼び出していないため、実際の利用一覧には含めない。\n'
        nb.metadata['clean_replay_requested']=True
    project_manager.enqueue_write(handle,prepare_clean_replay)
print('保存済み全10コードセルの出力をクリアしました。以後は新しいカーネルで順番に実行します。')

保存済み全10コードセルの出力をクリアしました。以後は新しいカーネルで順番に実行します。


In [25]:
with phase('MCP同期への書込み競合回避',write=True):
    def stage_insights(nb):
        final_index=next(i for i,c in enumerate(nb.cells) if c.metadata.get('role')=='final')
        nb.cells=nb.cells[:final_index+1]
        final=nb.cells[-1]
        final.source=final.source.replace('import nbformat\n','import nbformat, copy\n',1)
        anchor="    ledger_cell=next(c for c in snapshot.cells if c.metadata.get('role')=='ledger')\n"
        insertion="    evidence_handle=project_manager.ProjectHandle(handle.name,handle.root,DATA_DIR/'insight-evidence-validation.ipynb',DATA_DIR)\n    project_manager.ensure_notebook(evidence_handle)\n    def seed_evidence(nb):\n        nb.cells=[copy.deepcopy(initial_cell),copy.deepcopy(ledger_cell)]\n    project_manager.enqueue_write(evidence_handle,seed_evidence)\n"
        final.source=final.source.replace(anchor,anchor+insertion)
        final.source=final.source.replace('insight_engine.record_insight(handle,','insight_engine.record_insight(evidence_handle,')
        old="    for slot,(key,text) in enumerate(claims):\n        insight_engine.record_insight(evidence_handle,text,ledger_cell.execution_count,cited(key),'associational',language='ja')\n        def tag_latest(nb):\n            generated=nb.cells.pop()\n            generated.metadata.update({'generated_insight':True,'insight_slot':slot})\n            index=next(i for i,c in enumerate(nb.cells) if c.metadata.get('insight_slot')==slot)\n            nb.cells[index]=generated\n        project_manager.enqueue_write(handle,tag_latest)"
        new="    new_insights=[]\n    for slot,(key,text) in enumerate(claims):\n        insight_engine.record_insight(evidence_handle,text,ledger_cell.execution_count,cited(key),'associational',language='ja')\n        generated=copy.deepcopy(nbformat.read(evidence_handle.notebook_path,as_version=4).cells[-1])\n        generated.metadata.update({'generated_insight':True,'insight_slot':slot})\n        new_insights.append(generated)"
        assert old in final.source
        final.source=final.source.replace(old,new)
        anchor="        assert nb.cells[-1].metadata.get('role')=='final', '最終セルの位置が変わった'\n"
        inserted="        for slot,generated in enumerate(new_insights):\n            index=next(i for i,c in enumerate(nb.cells) if c.metadata.get('insight_slot')==slot)\n            generated.id=nb.cells[index].id\n            nb.cells[index]=generated\n"
        final.source=final.source.replace(anchor,anchor+inserted)
        final.source=final.source.replace("    improvements=[stream_issue,", "    sync_issue={'classification':'MCP連携・書込み同期候補（責任層未確定）','reproduction':'MCPで開いたNotebookにrecord_insight追加とenqueue_writeで削除・置換を短時間に反復','expected':'単一の最終状態のみが保存され重複セルが生じない','actual':'監査時26セルだったが再読込みで末尾にInsightが1つ増え27セルになった','impact':'監査後のセル構造・根拠が変わる可能性','workaround':'非アクティブの証拠検証Notebookに実出力を複製してrecord_insightを実行し、検証済み8セルを一度のenqueue_writeで固定IDのスロットへ反映','related_cells':['final'],'log':'作成用Notebookの同期回避セルと実行前後のread_notebook結果。Kaggle API・ベンチマーク問題とは別で、MCPのファイル監視とJupytermind書込み方式の責任分離は未完了'}\n    improvements=[stream_issue,sync_issue,")
        for c in nb.cells:
            if c.cell_type=='markdown' and c.source.startswith('# 使用したJupytermindモジュール'):
                c.source=c.source.replace('resolve_project, ensure_notebook','ProjectHandle, resolve_project, ensure_notebook')
            if c.cell_type=='markdown' and c.source.startswith('# 根拠付きInsightと再実行・監査'):
                c.source += '\n\nMCPのアクティブNotebookへの短時間反復書込みで末尾セル重複が観測されたため、実行済み証拠セルを非アクティブ検証Notebookへ複製しrecord_insightで検証した後、8個の固定IDスロットへ一括反映する。これは新しい計算結果を作るものではなく、実出力のコピーで検証する書込み上の回避策である。\n'
    project_manager.enqueue_write(handle,stage_insights)
print('検証済みInsightを単一書込みで反映するよう変更。全10コードセルは未実行状態のまま保持しました。')

検証済みInsightを単一書込みで反映するよう変更。全10コードセルは未実行状態のまま保持しました。


In [7]:
import ast, nbformat
lifecycle.mark_failed(RUN_ID)
print('再実行で分析者による表示名修正の括弧ミスを検出：failed。Jupytermind不具合ではありません。')
lifecycle.register_run(RUN_ID,handle.notebook_path)
with phase('表示名修正と全コード構文検査',write=True):
    def fix_syntax(nb):
        for c in nb.cells:
            if c.metadata.get('role')=='iteration2':
                c.source=c.source.replace("else 'gender'))))", "else 'gender')))")
            if c.metadata.get('role')=='initial':
                c.source=c.source.replace("s.split('(')[0]+': '", "s.split(',')[0].removeprefix('C(')+': '")
            if c.metadata.get('role')=='final':
                c.source=c.source.replace('Kaggle API失敗なし、', 'クリーン再実行の1回目で表示名修正に起因するSyntaxErrorを検出しfailed記録後、括弧を修正して全コード構文を検査、改めて先頭から再実行した。Kaggle API失敗なし、')
            if c.cell_type=='code':
                ast.parse(c.source)
                c.execution_count=None; c.outputs=[]
        nb.metadata['replay_retry_reason']='分析者の表示名変更時の括弧ミス。修正済み、全コードast.parse通過。'
    project_manager.enqueue_write(handle,fix_syntax)
print('10コードセルの構文検査を通過。初回基準45値はそのまま保持しました。')

再実行で分析者による表示名修正の括弧ミスを検出：failed。Jupytermind不具合ではありません。
10コードセルの構文検査を通過。初回基準45値はそのまま保持しました。


In [8]:
with phase('実際に呼び出したライフサイクル関数の記録更新',write=True):
    def update_usage(nb):
        for c in nb.cells:
            if c.cell_type=='markdown' and c.source.startswith('# 使用したJupytermindモジュール'):
                c.source=c.source.replace('mark_completed, get_run_status', 'mark_completed, mark_failed（作成用Notebookの再実行失敗記録）, get_run_status')
                c.source=c.source.replace('`lifecycle.mark_failed` は失敗分岐に記述したが成功した分析・再実行では呼び出していないため、実際の利用一覧には含めない。', '`lifecycle.mark_failed` はクリーン再実行で分析者コードのSyntaxErrorを検出した際に作成用Notebookから直接呼び出した。修正後に同run_idを再登録し、最後にcompletedを記録する。')
    project_manager.enqueue_write(handle,update_usage)
print('利用記録を実際の呼出しに合わせました。')

利用記録を実際の呼出しに合わせました。


In [9]:
import ast, nbformat
with warnings.catch_warnings(record=True) as font_repro_warnings:
    warnings.simplefilter('always')
    repro_png=visualization.render_chart(bar_data,kind='bar',x='preparation',y='3科目平均',title='日本語フォント再描画',xlabel='準備講座',ylabel='得点')
font_repro=[str(w.message) for w in font_repro_warnings if 'Glyph' in str(w.message) and 'missing' in str(w.message)]
print('FONT_CONTEXT_REPRO:', {'missing_glyph_warning_count':len(font_repro),'examples':font_repro[:3],'run_state':lifecycle.get_run_status(RUN_ID).state})
assert font_repro, '再現条件を確認できなかった'
(DATA_DIR/'font-context-reproduction.json').write_text(json.dumps({'glyph_warnings':font_repro,'condition':'初回render_chartをrc_contextで実行し、終了後に再び日本語render_chartを呼ぶ'},ensure_ascii=False,indent=2),encoding='utf-8')
lifecycle.register_run(RUN_ID,handle.notebook_path)
with phase('フォント状態の再現性修正',write=True):
    def fix_font_context(nb):
        for c in nb.cells:
            if c.metadata.get('role')=='charts':
                c.source=c.source.replace("plt.rc_context({'figure.autolayout': True})", "plt.rc_context({'figure.autolayout': True, 'font.family': 'IPAexGothic'})")
            if c.metadata.get('role')=='final':
                anchor='    improvements=[stream_issue,sync_issue,'
                replacement="    font_issue={'classification':'可視化・日本語フォントの状態依存','reproduction':'初回render_chartをmatplotlib rc_context内で実行し、終了後に日本語render_chartを再実行','expected':'毎回日本語フォントを設定し欠落グリフを発生させない','actual':'初回呼出しのフォント設定がrc_context終了で戻るが内部の初回フラグは真のままで、次回はフォント再設定されずGlyph missing警告が出る。新しいカーネルで初回から再実行して検出','impact':'以前の描画履歴次第で図が文字化けし再現性を損なう','workaround':'各描画rc_contextにfont.family=IPAexGothicを明示。render_chart初回で同梱フォントを登録してからカスタム図を描く','related_cells':['charts','final'],'log':'data/font-context-reproduction.json と作成用NotebookのFONT_CONTEXT_REPRO'}\n    improvements=[stream_issue,sync_issue,font_issue,"
                assert anchor in c.source
                c.source=c.source.replace(anchor,replacement)
                c.source=c.source.replace('Kaggle API失敗なし、', '再実行2回目でrc_contextと日本語フォント初回フラグの状態依存を検出しfailedとなったため、全描画でフォントを明示して再実行した。Kaggle API失敗なし、')
            if c.cell_type=='code':
                ast.parse(c.source);c.execution_count=None;c.outputs=[]
        nb.metadata['replay_retry_reason']='表示名SyntaxErrorとrc_context後のフォント状態依存を修正。最終検証は新カーネル全コードセル再実行。'
    project_manager.enqueue_write(handle,fix_font_context)
print('日本語フォントを各図で明示し、全10セルを再実行準備しました。')

FONT_CONTEXT_REPRO: {'missing_glyph_warning_count': 25, 'examples': ['Glyph 24471 (\\N{CJK UNIFIED IDEOGRAPH-5F97}) missing from font(s) DejaVu Sans.', 'Glyph 28857 (\\N{CJK UNIFIED IDEOGRAPH-70B9}) missing from font(s) DejaVu Sans.', 'Glyph 26085 (\\N{CJK UNIFIED IDEOGRAPH-65E5}) missing from font(s) DejaVu Sans.'], 'run_state': 'failed'}
日本語フォントを各図で明示し、全10セルを再実行準備しました。


In [14]:
with phase('保存後の独立notebook_audit'):
    completed_snapshot=nbformat.read(handle.notebook_path,as_version=4)
    completed_codes=[c for c in completed_snapshot.cells if c.cell_type=='code']
    actual_counts=[c.execution_count for c in completed_codes]
    assert actual_counts==list(range(1,11)), '保存済み全10セルの実行順序が一致しない'
    assert len(completed_snapshot.cells)==26 and sum(bool(c.metadata.get('generated_insight')) for c in completed_snapshot.cells)==8, 'セルの重複・欠落がある'
    completed_audit=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    assert completed_audit.ok and completed_audit.executed_code_cell_count==10 and not completed_audit.findings and not completed_audit.visual_findings
    assert reproduction['baseline_comparison']=='passed' and reproduction['max_absolute_difference']==0
    reproduction.update({'clean_kernel_top_to_bottom':True,'verification_basis':'保存後の全コードセルの実execution_count（実行中のIPython内部カウンタではない）','actual_saved_execution_counts':actual_counts,'all_code_cells_executed':True,'post_execution_audit_ok':True,'code_source_sha256':{c.metadata['role']:hashlib.sha256(c.source.encode()).hexdigest() for c in completed_codes}})
    print('保存後最終検証:',json.dumps({'reproducibility':reproduction,'audit':{**asdict(completed_audit),'ok':completed_audit.ok}},ensure_ascii=False,indent=2))
with phase('保存後検証のNotebook記録',write=True):
    def persist_completed_verification(nb):
        nb.metadata['reproducibility']=reproduction
        nb.metadata['audit']={**asdict(completed_audit),'ok':completed_audit.ok}
        nb.metadata['lifecycle_final']=asdict(lifecycle.get_run_status(RUN_ID))
        documentation=next(c for c in nb.cells if c.cell_type=='markdown' and c.source.startswith('# 根拠付きInsightと再実行・監査'))
        documentation.source += '\n\n## 保存完了後の最終検証（作成用NotebookのJupyter MCP実行セル）\n\n新しいカーネルで全10コードセルを上から再実行した。保存された実execution_countは `1,2,3,4,5,6,7,8,9,10`。初回と比較した45推定値の最大絶対差は `0.0`、CSVのSHA-256も一致した。完了後の `notebook_audit(visual_audit=True)` は **ok=True、実行済み10/10、未実行・エラー・根拠不整合・視覚的指摘すべて0**。Insightは8件、画像は3枚、セル重複なし。\n\n最終セル実行中の `get_ipython().execution_count` はツール保存値より先へ進むことがあり、セル内の暫定 `clean_kernel_top_to_bottom=False` はその内部カウンタ比較による。これは結果不一致ではなく、上記の保存後実execution_count検証を最終判定とする。検証コードと出力は `benchmark/students-bootstrap-v020-exp-10.ipynb`、監査報告・再現性判定・コードのSHA-256はNotebook metadataと `data/reproducibility-verification.json` に記録した。\n\nライフサイクル最終状態は **completed**、実行数・書込み数・ロック数はすべて0。推測・不明なデータ定義、独立性、合成指標の妥当性、母集団への一般化・因果同定の不成立という前提検査の警告は解消したと扱わず、上の限界と分析前提manifestを適用する。\n'
    project_manager.enqueue_write(handle,persist_completed_verification)
    (DATA_DIR/'reproducibility-verification.json').write_text(json.dumps(reproduction,ensure_ascii=False,indent=2),encoding='utf-8')
    (DATA_DIR/'notebook-audit-final.json').write_text(json.dumps({**asdict(completed_audit),'ok':completed_audit.ok},ensure_ascii=False,indent=2),encoding='utf-8')
lifecycle.mark_completed(RUN_ID)
final_status=lifecycle.wait_for_quiescence(RUN_ID)
assert final_status.state=='completed' and final_status.active_cell_executions==final_status.pending_notebook_writes==final_status.locks_held==0
with phase('最終ライフサイクル状態の保存',write=True):
    def persist_status(nb):
        nb.metadata['lifecycle_final']=asdict(final_status)
    project_manager.enqueue_write(handle,persist_status)
(DATA_DIR/'lifecycle-log.json').write_text(json.dumps({'events':phase_log,'final_status':asdict(lifecycle.get_run_status(RUN_ID))},ensure_ascii=False,indent=2),encoding='utf-8')
# Recheck the exact persistent artifact, after all its final writes.
post_write_audit=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
assert post_write_audit.ok and not post_write_audit.findings and not post_write_audit.visual_findings
assert [c.execution_count for c in nbformat.read(handle.notebook_path,as_version=4).cells if c.cell_type=='code']==list(range(1,11))
print('FINAL_RESULT:',{'notebook':str(handle.notebook_path),'audit_ok':post_write_audit.ok,'executed_code_cells':post_write_audit.executed_code_cell_count,'insights':post_write_audit.insight_cell_count,'visual_findings':len(post_write_audit.visual_findings),'lifecycle':asdict(lifecycle.get_run_status(RUN_ID))})

保存後最終検証: {
  "reproducibility": {
    "baseline_comparison": "passed",
    "metric_count": 45,
    "max_absolute_difference": 0.0,
    "rtol": 1e-08,
    "atol": 1e-08,
    "sha256_match": true,
    "clean_kernel_top_to_bottom": true,
    "previous_code_execution_counts": [
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9
    ],
    "current_execution_count": 11,
    "verification_basis": "保存後の全コードセルの実execution_count（実行中のIPython内部カウンタではない）",
    "actual_saved_execution_counts": [
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10
    ],
    "all_code_cells_executed": true,
    "post_execution_audit_ok": true,
    "code_source_sha256": {
      "setup": "838fbba7b82a0ea14623113dd478f696a0d901376fea6bc240b0109b4c1c48a5",
      "acquisition": "d47e0bbfc0353169805ff71a90530997e6d8bfe16f91b6a2200e922ed1acdd2f",
      "quality": "480078e39285317f60e0c840815c68cf40e5509e328846096834747b6a4a483d",
      "initial": "190598dd84

In [13]:
lifecycle.register_run(RUN_ID,handle.notebook_path)
with phase('MCP完了保存後の根拠manifest再同期',write=True):
    validated=nbformat.read(evidence_handle.notebook_path,as_version=4)
    validated_insights=[c for c in validated.cells if c.cell_type=='markdown']
    assert len(validated_insights)==8
    assert all('"execution_count":9' in c.source for c in validated_insights)
    def sync_after_mcp_save(nb):
        for slot,generated in enumerate(validated_insights):
            index=next(i for i,c in enumerate(nb.cells) if c.metadata.get('insight_slot')==slot)
            old_id=nb.cells[index].id
            nb.cells[index]=copy.deepcopy(generated)
            nb.cells[index].id=old_id
            nb.cells[index].metadata.update({'generated_insight':True,'insight_slot':slot})
        candidate_section=next(c for c in nb.cells if c.cell_type=='markdown' and c.source.startswith('# Jupytermind改善候補'))
        candidate_section.source+='\n\n## 保存後監査で確認したMCP連携の追加事象\nセル内監査では8Insightの根拠が新しいexecution_count=9に更新されていたが、MCPがセル完了時にアクティブNotebookを保存すると旧execution_count=16に戻り、保存後監査で8件のstale evidenceを検出した。分析結果自体は全45値一致し、全コードセルの実行順序も1〜10。これはMCPのセル完了保存とカーネル内enqueue_writeとの相互作用であり、Kaggleやベンチマークの問題ではない。Jupytermind単体の責任は未確定として連携改善候補に含める。回避策：対象Notebookを非アクティブにした後、検証Notebookのrecord_insight済みセルを一括同期して独立auditを実行する。作成用Notebookの「MCP完了保存後の根拠manifest再同期」セルと直前の監査エラーログを参照。\n'
        nb.metadata['mcp_post_save_evidence_sync']={'stale_count_detected':16,'actual_ledger_count':9,'restored_insight_count':8,'responsibility':'MCPセル保存とenqueue_writeの連携（未確定）'}
    project_manager.enqueue_write(handle,sync_after_mcp_save)
print('対象Notebookを非アクティブにした状態で、実際に検証済みの根拠count=9を同期しました。')

対象Notebookを非アクティブにした状態で、実際に検証済みの根拠count=9を同期しました。
